In [1]:
import numpy as np, pandas as pd, os
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
BT  = "/home/admins/rebuild_workspace/models_buildup_test"
OUT = "/home/admins/lip_codebase_clean/docs/results_buildup"
os.makedirs(OUT, exist_ok=True)
word_classes = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']

# pool the three seeds
y_true = np.concatenate([np.load(f"{BT}/y_true_b2_seed{s}.npy") for s in [11,22,33]])
y_pred = np.concatenate([np.argmax(np.load(f"{BT}/y_prob_b2_seed{s}.npy"), axis=1)
                         for s in [11,22,33]])

print("pooled samples:", len(y_true))
print("agreement:", (y_true == y_pred).mean())

cm = pd.DataFrame(confusion_matrix(y_true, y_pred),
                  index=word_classes, columns=word_classes)
cm.index.name, cm.columns.name = "Actual", "Predicted"
print(cm)
print()
print(classification_report(y_true, y_pred, target_names=word_classes,
                            digits=4, zero_division=0))

cm.to_csv(f"{OUT}/confusion_matrix_b2_pooled.csv")
rep = classification_report(y_true, y_pred, target_names=word_classes,
                            digits=4, output_dict=True, zero_division=0)
pd.DataFrame(rep).transpose().to_csv(f"{OUT}/classification_report_b2_pooled.csv")

fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, square=True,
            linewidths=0.5, linecolor='lightgray', ax=ax)
ax.set_title('Confusion Matrix — b2 on d138, 3 seeds pooled (600 predictions)')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_matrix_b2_pooled.png", dpi=200)
plt.close()

hist = [pd.read_csv(f"{BT}/history_b2_seed{s}.csv") for s in [11,22,33]]
fig, axes = plt.subplots(1, 2, figsize=(13,5))
for i, h in enumerate(hist):
    axes[0].plot(h.index+1, h['accuracy'], alpha=0.5, label=f'train seed {[11,22,33][i]}')
    axes[0].plot(h.index+1, h['val_accuracy'], ls='--', label=f'val seed {[11,22,33][i]}')
    axes[1].plot(h.index+1, h['loss'], alpha=0.5)
    axes[1].plot(h.index+1, h['val_loss'], ls='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].set_title('b2 on d138 — accuracy')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].set_title('b2 on d138 — loss')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT}/training_curves_b2.png", dpi=200)
plt.close()
print("saved")

pooled samples: 600
agreement: 0.71
Predicted  bat  cup  drop  eat  fish  hot  jump  milk  pen  red
Actual                                                         
bat         37    2     1    2     0    0     0     0   11    7
cup          1   47     1    0     0    2     4     0    0    5
drop         0    3    55    0     0    2     0     0    0    0
eat          3    0     0   46     0    0     0     0    5    6
fish         1    0     0    0    56    1     0     1    0    1
hot          0    0     7    0     0   52     1     0    0    0
jump         0   16     7    3     0    0    34     0    0    0
milk         0    0     0    0     4    0     0    31   25    0
pen         11    0     0    0     0    0     0     2   36   11
red          0    0     0   18     0    0     0     0   10   32

              precision    recall  f1-score   support

         bat     0.6981    0.6167    0.6549        60
         cup     0.6912    0.7833    0.7344        60
        drop     0.7746    0.916